# 🐘 AWS RDS PostgreSQL 실습 — Google Colab

**실습 목표** (PDF 강의안 기반)
1. AWS RDS PostgreSQL 17 인스턴스 자동 생성 (boto3)
2. VPC 보안 그룹 포트 5432 인바운드 규칙 자동 설정
3. psycopg2 로 RDS 연결 확인
4. dvdrental 샘플 데이터베이스 복원
5. 기초 SQL 실습 (테이블 조회, 집계, 조인)

---
**아키텍처**
```
Colab (Python/psycopg2)
    │
    │  port 5432
    ▼
AWS RDS PostgreSQL 17
  (db.t3.micro, 단일 AZ, 퍼블릭 액세스 활성화)
    │
    ▼
dvdrental DB  ← dvdrental.tar 복원
```
---
> ⚠️ **비용 주의**: db.t3.micro는 AWS 프리티어 12개월 무료 (750시간/월).
> 실습 후 반드시 **Cell 12** 의 인스턴스 삭제 코드를 실행하세요.

## Cell 1 — 패키지 설치

In [ ]:
!pip install boto3 psycopg2-binary pandas sqlalchemy -q
print('✅ 패키지 설치 완료')

## Cell 2 — AWS 인증 (Colab 비밀 사용)

> **Colab 비밀 설정 방법**
> 1. 왼쪽 사이드바 🔑 (Secrets) 클릭
> 2. 아래 키 이름으로 값 추가:
>    - `AWS_ACCESS_KEY_ID`
>    - `AWS_SECRET_ACCESS_KEY`
>    - `AWS_DEFAULT_REGION` (예: `ap-northeast-2`)
>    - `DB_PASSWORD` (설정할 RDS 마스터 비밀번호 — 8자 이상)

In [ ]:
from google.colab import userdata
import boto3
import time, json, requests

AWS_ACCESS_KEY_ID     = userdata.get('AWS_ACCESS_KEY_ID')
AWS_SECRET_ACCESS_KEY = userdata.get('AWS_SECRET_ACCESS_KEY')
AWS_REGION            = userdata.get('AWS_DEFAULT_REGION') or 'ap-northeast-2'
DB_PASSWORD           = userdata.get('DB_PASSWORD')

DB_INSTANCE_ID = 'postgresql-lab-01'
DB_NAME        = 'postgres'
DB_USERNAME    = 'postgres'
DB_PORT        = 5432
DB_CLASS       = 'db.t3.micro'
DB_ENGINE      = 'postgres'
DB_VERSION     = '17.4'
STORAGE_GB     = 20

session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=AWS_REGION
)
rds = session.client('rds')
ec2 = session.client('ec2')
print(f'✅ AWS 인증 완료 — 리전: {AWS_REGION}')

## Cell 3 — 현재 IP 확인 (보안 그룹 인바운드 설정용)

In [ ]:
my_ip = requests.get('https://api.ipify.org').text.strip()
MY_CIDR = f'{my_ip}/32'
print(f'🌐 Colab 외부 IP: {my_ip}')
print(f'   → 보안 그룹 인바운드 소스: {MY_CIDR}')

## Cell 4 — VPC 보안 그룹 포트 5432 추가 (PDF p.16~17)

In [ ]:
def setup_security_group_for_postgres(my_cidr):
    vpcs = ec2.describe_vpcs(Filters=[{'Name':'isDefault','Values':['true']}])
    vpc_id = vpcs['Vpcs'][0]['VpcId']
    print(f'📌 기본 VPC ID: {vpc_id}')
    sgs = ec2.describe_security_groups(Filters=[
        {'Name':'vpc-id','Values':[vpc_id]},
        {'Name':'group-name','Values':['default']}
    ])
    sg_id = sgs['SecurityGroups'][0]['GroupId']
    print(f'🔒 보안 그룹 ID: {sg_id}')
    existing = sgs['SecurityGroups'][0].get('IpPermissions', [])
    already = any(
        p.get('FromPort') == 5432 and
        any(r.get('CidrIp') == my_cidr for r in p.get('IpRanges', []))
        for p in existing
    )
    if already:
        print(f'✅ 포트 5432 ({my_cidr}) 인바운드 규칙 이미 존재')
    else:
        ec2.authorize_security_group_ingress(
            GroupId=sg_id,
            IpPermissions=[{
                'IpProtocol':'tcp','FromPort':5432,'ToPort':5432,
                'IpRanges':[{'CidrIp':my_cidr,'Description':'PostgreSQL from Colab'}]
            }]
        )
        print(f'✅ 포트 5432 인바운드 규칙 추가 완료 (소스: {my_cidr})')
    return sg_id

sg_id = setup_security_group_for_postgres(MY_CIDR)
print(f'보안 그룹 설정 완료: {sg_id}')

## Cell 5 — RDS PostgreSQL 인스턴스 생성 (PDF p.5~15)

In [ ]:
def create_rds_postgresql(sg_id):
    try:
        resp = rds.describe_db_instances(DBInstanceIdentifier=DB_INSTANCE_ID)
        inst = resp['DBInstances'][0]
        print(f'ℹ️  기존 인스턴스: {DB_INSTANCE_ID} ({inst["DBInstanceStatus"]})')
        return inst
    except rds.exceptions.DBInstanceNotFoundFault:
        pass
    print(f'🚀 RDS PostgreSQL {DB_VERSION} 생성 (클래스: {DB_CLASS}, 스토리지: {STORAGE_GB}GB)')
    rds.create_db_instance(
        DBInstanceIdentifier=DB_INSTANCE_ID,
        DBInstanceClass=DB_CLASS,
        Engine=DB_ENGINE,
        EngineVersion=DB_VERSION,
        MasterUsername=DB_USERNAME,
        MasterUserPassword=DB_PASSWORD,
        DBName=DB_NAME,
        AllocatedStorage=STORAGE_GB,
        StorageType='gp2',
        PubliclyAccessible=True,
        MultiAZ=False,
        AutoMinorVersionUpgrade=True,
        BackupRetentionPeriod=1,
        VpcSecurityGroupIds=[sg_id],
        Tags=[{'Key':'Purpose','Value':'PostgreSQL-Lab'}]
    )
    print('⏳ 생성 요청 완료 — 약 5~10분 후 사용 가능')
    return None

inst = create_rds_postgresql(sg_id)

## Cell 6 — available 상태 대기 및 엔드포인트 확인 (PDF p.15)

In [ ]:
def wait_for_rds_available(instance_id, timeout_min=20):
    print(f'⏳ RDS 상태 대기 중 (최대 {timeout_min}분)...')
    deadline = time.time() + timeout_min * 60
    while time.time() < deadline:
        resp = rds.describe_db_instances(DBInstanceIdentifier=instance_id)
        inst = resp['DBInstances'][0]
        status = inst['DBInstanceStatus']
        print(f'   ��태: {status}', end='')
        if status == 'available':
            ep = inst['Endpoint']['Address']
            print(f'\n✅ 사용 가능! 엔드포인트: {ep}')
            return ep
        print(' — 30초 후 재확인...')
        time.sleep(30)
    raise TimeoutError('시간 초과')

RDS_HOST = wait_for_rds_available(DB_INSTANCE_ID)
print(f'Host: {RDS_HOST} | Port: {DB_PORT} | User: {DB_USERNAME}')

## Cell 7 — psycopg2 연결 확인 (PDF p.21~22)

In [ ]:
import psycopg2
import pandas as pd

def get_connection(dbname=DB_NAME):
    return psycopg2.connect(
        host=RDS_HOST, port=DB_PORT, dbname=dbname,
        user=DB_USERNAME, password=DB_PASSWORD, connect_timeout=10
    )

try:
    conn = get_connection()
    cur = conn.cursor()
    cur.execute('SELECT version();')
    version = cur.fetchone()[0]
    cur.close(); conn.close()
    print('✅ RDS PostgreSQL 연결 성공!')
    print(f'   버전: {version}')
except Exception as e:
    print(f'❌ 연결 실패: {e}')
    print('   → 보안 그룹 5432 포트 및 엔드포인트를 확인하세요.')

## Cell 8 — dvdrental DB 생성 및 복원 (PDF p.23~25)

dvdrental: DVD 대여점 샘플 DB — 15개 테이블, 21,000+ 레코드

In [ ]:
import subprocess, os, urllib.request

# dvdrental.zip ��운로드 및 압축 해제
DVDRENTAL_URL = 'https://www.postgresqltutorial.com/wp-content/uploads/2019/05/dvdrental.zip'
ZIP_PATH = '/tmp/dvdrental.zip'
TAR_PATH = '/tmp/dvdrental.tar'
print('📥 dvdrental 다운로드...')
urllib.request.urlretrieve(DVDRENTAL_URL, ZIP_PATH)
subprocess.run(['unzip','-o',ZIP_PATH,'-d','/tmp/'], capture_output=True)
print(f'✅ 완료: {TAR_PATH}')

# dvdrental 데이터베이스 생성
conn = get_connection(); conn.autocommit = True; cur = conn.cursor()
cur.execute("SELECT 1 FROM pg_database WHERE datname='dvdrental';")
if cur.fetchone():
    print('ℹ️  dvdrental DB 이미 존재')
else:
    cur.execute('CREATE DATABASE dvdrental OWNER postgres;')
    print('✅ dvdrental 데이터베이스 생성 완료')
cur.close(); conn.close()

# postgresql-client 설치 후 pg_restore 실행
!apt-get install -y postgresql-client -q
os.environ['PGPASSWORD'] = DB_PASSWORD
result = subprocess.run([
    'pg_restore','--host',RDS_HOST,'--port',str(DB_PORT),
    '--username',DB_USERNAME,'--dbname','dvdrental',
    '--no-password','--verbose', TAR_PATH
], capture_output=True, text=True)
if result.returncode == 0 or 'already exists' in result.stderr:
    print('✅ dvdrental 복원 완료!')
else:
    print(f'⚠️  출력:\n{result.stderr[:800]}')

## Cell 9 — 기초 SQL 실습 (PDF p.26~27)

In [ ]:
def run_query(sql, conn=None):
    _conn = conn or get_connection('dvdrental')
    df = pd.read_sql_query(sql, _conn)
    if not conn: _conn.close()
    return df

dv_conn = get_connection('dvdrental')

# 9-1. 테이블 목록
print('='*50); print('📋 9-1. dvdrental 테이블 목록'); print('='*50)
display(run_query("""
    SELECT table_name, table_type FROM information_schema.tables
    WHERE table_schema='public' ORDER BY table_type, table_name;
""", dv_conn))

# 9-2. 영화 목록 상위 10건
print('='*50); print('🎬 9-2. 영화 목록 (film — 상위 10건)'); print('='*50)
display(run_query("""
    SELECT film_id, title, release_year, rental_rate, rating, length AS duration_min
    FROM film ORDER BY film_id LIMIT 10;
""", dv_conn))

# 9-3. 카테고리별 집계
print('='*50); print('📊 9-3. 카테고리별 영화 수 (GROUP BY)'); print('='*50)
display(run_query("""
    SELECT c.name AS category, COUNT(f.film_id) AS film_count,
           ROUND(AVG(f.rental_rate),2) AS avg_rate
    FROM category c
    JOIN film_category fc ON c.category_id=fc.category_id
    JOIN film f ON fc.film_id=f.film_id
    GROUP BY c.name ORDER BY film_count DESC;
""", dv_conn))

In [ ]:
# 9-4. 고객별 대여 Top 10 (JOIN)
print('='*50); print('👥 9-4. 고객별 대여 횟수 Top 10'); print('='*50)
display(run_query("""
    SELECT c.customer_id,
           c.first_name||' '||c.last_name AS full_name,
           c.email, COUNT(r.rental_id) AS total_rentals,
           SUM(p.amount)::numeric(10,2) AS total_paid
    FROM customer c
    JOIN rental r ON c.customer_id=r.customer_id
    JOIN payment p ON r.rental_id=p.rental_id
    GROUP BY c.customer_id,c.first_name,c.last_name,c.email
    ORDER BY total_rentals DESC LIMIT 10;
""", dv_conn))

# 9-5. 월별 매출
print('='*50); print('💰 9-5. 월별 매출 ���계'); print('='*50)
display(run_query("""
    SELECT TO_CHAR(payment_date,'YYYY-MM') AS month,
           COUNT(*) AS transactions,
           SUM(amount)::numeric(10,2) AS total_revenue,
           ROUND(AVG(amount),2) AS avg_payment
    FROM payment
    GROUP BY TO_CHAR(payment_date,'YYYY-MM') ORDER BY month;
""", dv_conn))

# 9-6. 배우별 출연 Top 10
print('='*50); print('🎭 9-6. 배우별 출연 영화 수 Top 10'); print('='*50)
display(run_query("""
    SELECT a.actor_id,
           a.first_name||' '||a.last_name AS actor_name,
           COUNT(fa.film_id) AS film_count
    FROM actor a
    JOIN film_actor fa ON a.actor_id=fa.actor_id
    GROUP BY a.actor_id,a.first_name,a.last_name
    ORDER BY film_count DESC LIMIT 10;
""", dv_conn))

dv_conn.close()
print('\n✅ 모든 SQL 실습 완료!')

## Cell 10 — 자유 SQL 실습

In [ ]:
MY_SQL = """
SELECT f.title, c.name AS category, f.rental_rate, f.rating
FROM film f
JOIN film_category fc ON f.film_id=fc.film_id
JOIN category c ON fc.category_id=c.category_id
WHERE f.rental_rate > 4.0
ORDER BY f.rental_rate DESC, f.title LIMIT 15;
"""
df_result = run_query(MY_SQL)
print(f'📋 결과 행 수: {len(df_result)}')
display(df_result)

## Cell 11 — pgAdmin4 연결 정보 출력 (PDF p.19~21)

In [ ]:
print('='*55)
print('  pgAdmin4 / DBeaver / psql 연결 정보')
print('='*55)
print(f'  Host      : {RDS_HOST}')
print(f'  Port      : {DB_PORT}')
print(f'  Username  : {DB_USERNAME}')
print(f'  Password  : (Colab Secret: DB_PASSWORD)')
print(f'  Database  : dvdrental')
print('='*55)
print(f'\npsql -h {RDS_HOST} -U {DB_USERNAME} -d dvdrental -p {DB_PORT}')
print(f'postgresql://{DB_USERNAME}:<pw>@{RDS_HOST}:{DB_PORT}/dvdrental')

## Cell 12 — RDS 인스턴스 삭제 ⚠️
> CONFIRM_DELETE = True 로 변경 후 실행하면 인스턴스가 삭제됩니다.

In [ ]:
CONFIRM_DELETE = False  # True 로 변경하면 삭제

if CONFIRM_DELETE:
    print(f'🗑️  RDS 삭제: {DB_INSTANCE_ID}')
    rds.delete_db_instance(
        DBInstanceIdentifier=DB_INSTANCE_ID,
        SkipFinalSnapshot=True,
        DeleteAutomatedBackups=True
    )
    print('⏳ 삭제 중 (약 5분)...')
    waiter = rds.get_waiter('db_instance_deleted')
    waiter.wait(DBInstanceIdentifier=DB_INSTANCE_ID,
                WaiterConfig={'Delay':20,'MaxAttempts':30})
    print('✅ 삭제 완료!')
else:
    print('ℹ️  삭제 건너뜀. CONFIRM_DELETE=True 로 변경 후 재실행하거나 AWS 콘솔에서 삭제하세요.')

## 📋 실습 요약

| Cell | 내용 | PDF 참조 |
|------|------|----------|
| 1 | 패키지 설치 (boto3, psycopg2, pandas) | — |
| 2 | Colab Secrets AWS 인증 | — |
| 3 | Colab 외부 IP 확인 | p.17 |
| 4 | 보안 그룹 포트 5432 자동 설정 | p.16~17 |
| 5 | RDS PostgreSQL 17 인스턴스 생성 | p.5~15 |
| 6 | available 상태 대기 + 엔드포인트 | p.15 |
| 7 | psycopg2 연결 확인 | p.21~22 |
| 8 | dvdrental DB 생성 + .tar 복원 | p.23~25 |
| 9 | 기초 SQL 실습 6개 쿼리 | p.26~27 |
| 10 | 자유 SQL 실습 | — |
| 11 | pgAdmin4 연결 정보 출력 | p.19~21 |
| 12 | 인스턴스 삭제 (비용 절약) | — |

**dvdrental ERD 핵심 관계**
```
actor ──< film_actor >── film ��─< film_category >── category
                          │
                    inventory
                          │
                       rental ──── customer
                          │
                       payment
```